# <center>**Phân tích luật kết hợp (Association Rule Mining) bằng thuật toán Apriori**<center>

Mục tiêu của bài thực hành này là sử dụng thuật toán Apriori để tìm ra các luật kết hợp (luật mua sắm) từ tập dữ liệu bán lẻ (Online Retail Dataset), giúp xác định các mặt hàng thường được mua cùng nhau.

In [3]:
# Cài đặt thư viện nếu chưa có (chạy trong terminal hoặc uncomment nếu cần)
!pip install apyori pandas requests

import pandas as pd
from apyori import apriori
import requests
from io import BytesIO
from zipfile import ZipFile

# Tải và xử lý dữ liệu từ link (UCI Online Retail dataset)
url = 'https://archive.ics.uci.edu/static/public/352/online+retail.zip'
response = requests.get(url)
zipfile = ZipFile(BytesIO(response.content))

# Đọc file Excel từ zip
online = pd.read_excel(zipfile.open('Online Retail.xlsx'), header=0)

  Preparing metadata (setup.py) ... done
  Created wheel for apyori: filename=apyori-1.1.2-py3-none-any.whl size=5954 sha256=2ab2bc30f51c5e977e4e2cc9ce7d1944eab1f6770be1358afddbbd689957a1d5
  Stored in directory: /root/.cache/pip/wheels/7f/49/e3/42c73b19a264de37129fadaa0c52f26cf50e87de08fb9804af
Successfully built apyori


## **Tiền xử lý dữ liệu**

In [4]:
# 1. Tạo cột IsCPresent để lọc hóa đơn hủy (bắt đầu bằng 'C')
online['IsCPresent'] = (online['InvoiceNo'].astype(str).apply(lambda x: 1 if x.find('C') != -1 else 0))

# 2. Lọc dữ liệu: Quantity > 0, không phải hóa đơn hủy, chỉ giữ cột cần thiết, drop NA
online1 = (online.loc[online["Quantity"] > 0]
           .loc[online['IsCPresent'] != 1]
           .loc[:, ["InvoiceNo", "Description"]]
           .dropna())

# 3. Giới hạn 5000 hóa đơn đầu tiên để giảm thời gian tính toán
invoice_no_list = online1.InvoiceNo.tolist()
invoice_no_list = list(set(invoice_no_list))
subset_invoice_no_list = invoice_no_list[0:5000]
online1 = online1.loc[online1["InvoiceNo"].isin(subset_invoice_no_list)]

# 4. Chuyển đổi dữ liệu thành list transactions (định dạng cần thiết cho Apriori)
invoice_item_list = []
for num in list(set(online1.InvoiceNo.tolist())):
    tmp_df = online1.loc[online1['InvoiceNo'] == num]
    tmp_items = tmp_df.Description.tolist()
    invoice_item_list.append(tmp_items)

print("Số lượng transactions:", len(invoice_item_list))
print("Ví dụ transaction đầu tiên:", invoice_item_list[0])

Số lượng transactions: 5000
Ví dụ transaction đầu tiên: ['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN', 'CREAM CUPID HEARTS COAT HANGER', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES', 'GLASS STAR FROSTED T-LIGHT HOLDER']


## **Áp dụng giải thuật Apriori**

Sử dụng hàm `apriori` để tìm các luật. Chúng ta thiết lập các ngưỡng sau:
* **min_support = 0.01 (1%)**: Tần suất xuất hiện tối thiểu của một tập hợp mặt hàng.
* **min_confidence = 0.6 (60%)**: Độ tin cậy tối thiểu của luật.
* **min_lift = 1.1**: Chỉ số Lift tối thiểu (chỉ ra mối quan hệ tích cực, không ngẫu nhiên).

In [6]:
# Áp dụng giải thuật Apriori
rules = apriori(transactions=invoice_item_list,
                min_support=0.01,
                min_confidence=0.6,
                min_lift=1.1,
                min_length=2,
                max_length=4)

## **Trích xuất và phân tích luật kết hợp**
Chuyển kết quả từ đối tượng Generator thành list, sau đó duyệt qua và in chi tiết từng luật với các chỉ số **Support**, **Confidence**, và **Lift**. Cuối cùng, lưu kết quả vào DataFrame để tiện cho việc phân tích.

In [7]:
# Chuyển kết quả thành list và hiển thị
output = list(rules)
print("Số lượng luật tìm thấy:", len(output))

# Hiển thị chi tiết từng luật (antecedents -> consequents)
for item in output:
    pair = item[0]
    items = [x for x in pair]
    # item[2] là list các mối quan hệ (luật) được tìm thấy từ itemset đó
    relation = item[2][0]

    antecedents = list(relation[0]) if relation[0] else []  # Phần trước (Antecedents)
    consequents = list(relation[1]) if relation[1] else []  # Phần sau (Consequents)

    print("Rule:", antecedents, "->", consequents)
    print("Support:", str(round(item[1], 4)))
    print("Confidence:", str(round(relation[2], 4)))
    print("Lift:", str(round(relation[3], 4)))
    print("=====================================")

# Lưu kết quả vào DataFrame
results = []
for item in output:
    pair = item[0]
    items = [x for x in pair]
    relation = item[2][0]

    results.append({
        "Rule (Antecedents -> Consequents)": f"{list(relation[0])} -> {list(relation[1])}",
        "Support": item[1],
        "Confidence": relation[2],
        "Lift": relation[3]
    })
df_results = pd.DataFrame(results)
print("\n--- 5 Luật Kết hợp hàng đầu ---")
print(df_results.sort_values(by='Lift', ascending=False).head())

Số lượng luật tìm thấy: 566
Rule: ['12 PENCILS SMALL TUBE SKULL'] -> ['12 PENCILS SMALL TUBE RED RETROSPOT']
Support: 0.0152
Confidence: 0.6847
Lift: 24.8074
Rule: ['ALARM CLOCK BAKELIKE CHOCOLATE'] -> ['ALARM CLOCK BAKELIKE GREEN']
Support: 0.014
Confidence: 0.6731
Lift: 12.3274
Rule: ['ALARM CLOCK BAKELIKE CHOCOLATE'] -> ['ALARM CLOCK BAKELIKE RED ']
Support: 0.014
Confidence: 0.6731
Lift: 12.9438
Rule: ['ALARM CLOCK BAKELIKE IVORY'] -> ['ALARM CLOCK BAKELIKE GREEN']
Support: 0.0164
Confidence: 0.6119
Lift: 11.2077
Rule: ['ALARM CLOCK BAKELIKE ORANGE'] -> ['ALARM CLOCK BAKELIKE GREEN']
Support: 0.0208
Confidence: 0.7482
Lift: 13.7033
Rule: ['ALARM CLOCK BAKELIKE GREEN'] -> ['ALARM CLOCK BAKELIKE RED ']
Support: 0.0358
Confidence: 0.6557
Lift: 12.6092
Rule: ['ALARM CLOCK BAKELIKE IVORY'] -> ['ALARM CLOCK BAKELIKE RED ']
Support: 0.0172
Confidence: 0.6418
Lift: 12.3421
Rule: ['ALARM CLOCK BAKELIKE ORANGE'] -> ['ALARM CLOCK BAKELIKE RED ']
Support: 0.0192
Confidence: 0.6906
Lift: 13.281

---
# <center>**END**<center>